# Homework: Bellman Updates in MountainCar and Acrobot

This notebook introduces you to two classic control environments: `MountainCar-v0` and `Acrobot-v1`.
Task — implement tabular Q-learning with Bellman updates, compare training strategies, and draw conclusions for each environment.

## Learning objectives
- develop discretizers for different continuous state spaces (2D for `MountainCar`, 6D for `Acrobot`)
- implement tabular Q-learning parameterized by the environment specification
- compare the effect of discretization and `epsilon` schedules across two independent experiments
- formulate recommendations for tuning Bellman-based algorithms for new environments

## Working format
- Work through the notebook top to bottom; cells with `TODO` should be filled in with code or text.
- If running the notebook in Colab, first install the dependencies (cell below).
- Record all key observations: plots, tables, numeric metrics.
- At the end, fill in the conclusions sections and answer the questions.


### Recommended workflow:
1. First implement the `Discretizer` class — it is the foundation for everything else
2. Test the discretizer on simple examples (create test cells)
3. Implement the `QLearningAgent` class methods one by one
4. Run a small training run (100-200 episodes) for debugging
5. Only then run the full experiments

### Installing dependencies
Run the cell below only in environments without the libraries pre-installed.

In [ ]:
# If you're working in Colab, uncomment the lines below.
# !pip install gymnasium numpy matplotlib tqdm -q

In [ ]:
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

In [ ]:
SEED = 2025
random.seed(SEED)
np.random.seed(SEED)

## 1A. Exploring the `MountainCar-v0` environment
Before discretizing, explore the state space and reward dynamics. Collect several episodes with a random
policy, record the min/max for each coordinate, and describe which states are visited most often.

In [ ]:
# Exploring the MountainCar-v0 environment
# Goal: understand the state ranges for correct discretization

mc_env = gym.make('MountainCar-v0')
mc_rollouts, mc_rewards, mc_lengths = [], [], []

# Collect 10 episodes with a random policy
for ep in range(10):
    state, _ = mc_env.reset(seed=SEED + ep)
    done = False
    total_reward = 0.0
    steps = 0

    while not done:
        # Save the current state for analysis
        mc_rollouts.append(state)

        # TODO: pick a random action from the environment's action space
        # Hint: use mc_env.action_space.sample()
        action = ...  # ← your code

        # Take a step in the environment
        state, reward, terminated, truncated, _ = mc_env.step(action)

        # TODO: update the counters
        # total_reward += ...
        # steps += ...
        # done = ... or ...

    mc_rewards.append(total_reward)
    mc_lengths.append(steps)

mc_env.close()

# Analyze the collected data
mc_rollouts = np.asarray(mc_rollouts)
position = mc_rollouts[:, 0]  # First coordinate — car position
velocity = mc_rollouts[:, 1]  # Second coordinate — velocity

print(f'Position: min={position.min():.3f}, max={position.max():.3f}')
print(f'Velocity: min={velocity.min():.3f}, max={velocity.max():.3f}')
print(f'Average reward of the random policy: {np.mean(mc_rewards):.1f}')
print(f'Average episode length: {np.mean(mc_lengths):.1f} steps')

## 1B. Exploring the `Acrobot-v1` environment
`Acrobot` has six features (cos/sin of angles and angular velocities). Explore the ranges and make sure you
understand the velocity limits. Record observations for which the episode terminates.

In [ ]:
# Exploring the Acrobot-v1 environment
# Acrobot is a two-link pendulum with 6 features:
# [cos(θ1), sin(θ1), cos(θ2), sin(θ2), θ1_dot, θ2_dot]

acro_env = gym.make('Acrobot-v1')
acro_rollouts, acro_rewards, acro_lengths = [], [], []

for ep in range(10):
    state, _ = acro_env.reset(seed=SEED + 100 + ep)
    done = False
    total_reward = 0.0
    steps = 0

    while not done:
        acro_rollouts.append(state)

        # TODO: pick a random action (similar to MountainCar)
        action = ...  # ← your code

        state, reward, terminated, truncated, _ = acro_env.step(action)

        # TODO: update the counters (similar to MountainCar)
        # ...

    acro_rewards.append(total_reward)
    acro_lengths.append(steps)

acro_env.close()

# Analyze the 6-dimensional state space
acro_rollouts = np.asarray(acro_rollouts)
mins = acro_rollouts.min(axis=0)
maxs = acro_rollouts.max(axis=0)

feature_names = ['cos(θ1)', 'sin(θ1)', 'cos(θ2)', 'sin(θ2)', 'θ1_dot', 'θ2_dot']
for i, (name, mn, mx) in enumerate(zip(feature_names, mins, maxs)):
    print(f'{name}: [{mn:.3f}, {mx:.3f}]')

print(f'\nAverage reward of the random policy: {np.mean(acro_rewards):.1f}')
print(f'Average episode length: {np.mean(acro_lengths):.1f} steps')

## 2. Environment specifications
To reuse code, let's describe each environment via an `EnvSpec`: observation ranges, default bins, and step count.
You can add your own specifications (e.g., for modified maps).

In [ ]:
@dataclass
class EnvSpec:
    name: str
    observation_ranges: Tuple[Tuple[float, float], ...]
    default_bins: Tuple[int, ...]
    max_steps: int
    reward_baseline: float
    description: str


ENV_SPECS: Dict[str, EnvSpec] = {
    "MountainCar-v0": EnvSpec(
        name="MountainCar-v0",
        observation_ranges=((-1.2, 0.6), (-0.07, 0.07)),
        default_bins=(24, 24),
        max_steps=200,
        reward_baseline=-110.0,
        description="2 dimensions: position and velocity, the goal is to reach the top",
    ),
    "Acrobot-v1": EnvSpec(
        name="Acrobot-v1",
        observation_ranges=((-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0), (-4.0, 4.0), (-9.0, 9.0)),
        default_bins=(8, 8, 8, 8, 12, 12),
        max_steps=500,
        reward_baseline=-100.0,
        description="6 features: cos/sin of the angles and angular velocities of the two links",
    ),
}

print("Available specifications:")
for spec in ENV_SPECS.values():
    print(f"- {spec.name}: dims={len(spec.observation_ranges)}, default_bins={spec.default_bins}")

## 3. Universal discretizer
Implement `Discretizer`, which accepts a list of ranges and the corresponding number of bins per dimension.
The class should be able to:
1. validate the input (`ranges` and `bins` of equal length, bins ≥ 1);
2. clip observations to the allowed ranges;
3. convert each dimension into a bin index (`np.digitize` or manual rules);
4. flatten indices into a single scalar for indexing into a Q-table of any size.

In [ ]:
class Discretizer:
    """
    Converts a continuous state into a discrete index.

    Idea: split each dimension into bins[i] equal intervals.
    For example, for the range [-1, 1] with 4 bins we get the intervals:
    [-1, -0.5), [-0.5, 0), [0, 0.5), [0.5, 1]
    """

    def __init__(self, ranges: Tuple[Tuple[float, float], ...], bins: Tuple[int, ...]):
        # Validate the input data
        if len(ranges) != len(bins):
            raise ValueError('ranges and bins must have the same length')
        if any(b < 1 for b in bins):
            raise ValueError('Number of bins must be ≥ 1')

        self.ranges = tuple(ranges)
        self.bins = tuple(bins)
        self.edges: List[np.ndarray] = []

        # Precompute bin edges for each dimension
        for (low, high), num_bins in zip(self.ranges, self.bins):
            if num_bins == 1:
                # A single bin — no edges
                self.edges.append(np.array([], dtype=np.float32))
            else:
                # TODO: create an array of (num_bins - 1) evenly spaced edges
                # Hint: np.linspace(low, high, num_bins - 1)
                # Example: for [-1, 1] and 4 bins → edges: [-0.5, 0.0, 0.5]
                edges = ...  # ← your code
                self.edges.append(edges)

    def clip(self, state: np.ndarray) -> np.ndarray:
        """Clips state values to the specified ranges."""
        state = np.asarray(state, dtype=np.float32)
        clipped = []
        for value, (low, high) in zip(state, self.ranges):
            # TODO: clip value to the range [low, high]
            # Hint: np.clip(value, low, high)
            clipped_value = ...  # ← your code
            clipped.append(float(clipped_value))
        return np.asarray(clipped, dtype=np.float32)

    def to_bin_indices(self, state: np.ndarray) -> Tuple[int, ...]:
        """Converts a state into a tuple of bin indices."""
        clipped = self.clip(state)
        indices: List[int] = []

        for value, edges, num_bins in zip(clipped, self.edges, self.bins):
            # TODO: find the bin index for value
            # Hint: np.digitize(value, edges) returns the bin number
            # Important: the result must be in the range [0, num_bins - 1]
            idx = ...  # ← your code (use np.digitize)
            idx = min(max(idx, 0), num_bins - 1)  # guard against out-of-range values
            indices.append(idx)

        return tuple(indices)

    def flat_index(self, indices: Tuple[int, ...]) -> int:
        """Converts a multi-dimensional index into a one-dimensional one (for the Q-table)."""
        # TODO: use np.ravel_multi_index to "flatten" the indices
        # Example: for bins=(3, 4), index (1, 2) → 1*4 + 2 = 6
        return ...  # ← your code

    @property
    def num_states(self) -> int:
        """Total number of discrete states."""
        # TODO: return the product of all bins
        # Hint: np.prod(self.bins)
        return ...  # ← your code

### Testing the discretizer

After implementing `Discretizer`, check its behavior with a simple example:

In [ ]:
# Testing the discretizer
# Uncomment after implementing the Discretizer class

# # Simple test: a 2D space with 3x4 bins
# test_disc = Discretizer(
#     ranges=((-1.0, 1.0), (-2.0, 2.0)),
#     bins=(3, 4)
# )
#
# # Check 1: total number of states
# print(f"Total states: {test_disc.num_states}")
# assert test_disc.num_states == 12, "Error: should be 3 * 4 = 12"
#
# # Check 2: clipping
# state = np.array([0.5, -3.0])  # -3.0 is outside the range [-2, 2]
# clipped = test_disc.clip(state)
# print(f"Clipped [{0.5}, {-3.0}] → {clipped}")
# assert clipped[1] == -2.0, "Error: should be clipped to -2.0"
#
# # Check 3: center indexing
# bin_indices = test_disc.to_bin_indices(np.array([0.0, 0.0]))
# print(f"Indices for [0.0, 0.0]: {bin_indices}")
# # For the center of the range we expect the middle indices: (1, 2)
#
# # Check 4: flat index
# flat = test_disc.flat_index((1, 2))
# print(f"Flat index for (1, 2): {flat}")
# # For bins=(3, 4): flat = 1*4 + 2 = 6
# assert flat == 6, "Error in flat_index"
#
# print("\n✅ Discretizer works correctly!")

## 4. Configuration and the Q-learning agent
Now let's generalize the agent. `QLearningConfig` holds the environment name, number of episodes, learning, and exploration parameters.
The agent should work with any specification from `ENV_SPECS`, using either the default bins or ones overridden in the config.

In [ ]:
@dataclass
class QLearningConfig:
    """Configuration for the Q-learning agent."""
    env_name: str = "MountainCar-v0"
    num_episodes: int = 4000
    max_steps: Optional[int] = None
    learning_rate: float = 0.1       # α — learning rate
    discount: float = 0.99           # γ — discount for future rewards
    epsilon_start: float = 1.0       # initial ε (full exploration)
    epsilon_end: float = 0.05        # final ε (nearly greedy policy)
    epsilon_decay_episodes: int = 2000  # number of episodes over which to decay ε
    bins: Optional[Tuple[int, ...]] = None
    seed: int = 42


class QLearningAgent:
    """
    Tabular Q-learning agent.

    Training algorithm (pseudocode):
    ─────────────────────────────
    1. Initialize Q(s, a) = 0 for all pairs
    2. For each episode:
       a. s ← initial state
       b. Repeat (for each step):
          i.   Choose a via ε-greedy: random with probability ε, otherwise argmax Q(s, ·)
          ii.  Take a, get r, s'
          iii. Update Q(s, a) ← Q(s, a) + α·[r + γ·max_a' Q(s', a') - Q(s, a)]
          iv.  s ← s'
       c. Decay ε
    """

    def __init__(self, env: gym.Env, config: QLearningConfig):
        if config.env_name not in ENV_SPECS:
            raise ValueError(f"Unknown environment: {config.env_name}")

        self.env = env
        self.config = config
        self.spec = ENV_SPECS[config.env_name]
        self.max_steps = config.max_steps or self.spec.max_steps
        bins = config.bins or self.spec.default_bins

        # Create the discretizer
        self.discretizer = Discretizer(self.spec.observation_ranges, bins)
        self.num_actions = env.action_space.n

        # Q-table: [num_states × num_actions]
        self.q_table = np.zeros((self.discretizer.num_states, self.num_actions), dtype=np.float32)
        self.rng = np.random.default_rng(config.seed)

    def epsilon_by_episode(self, episode: int) -> float:
        """
        Linear decay of epsilon from start to end.

        Formula: ε = start - (start - end) × min(episode / decay_episodes, 1)
        """
        decay_steps = max(1, self.config.epsilon_decay_episodes)
        # TODO: compute the current epsilon
        # frac = min(episode / decay_steps, 1.0)  # progress fraction [0, 1]
        # epsilon = start + frac * (end - start)  # linear interpolation
        frac = ...  # ← your code
        epsilon = ...  # ← your code
        return epsilon

    def state_index(self, obs: np.ndarray) -> int:
        """Converts an environment observation into a Q-table index."""
        # TODO: use the discretizer
        # 1. Get the bin indices: self.discretizer.to_bin_indices(obs)
        # 2. Convert to a flat index: self.discretizer.flat_index(...)
        return ...  # ← your code

    def select_action(self, state_idx: int, epsilon: float) -> int:
        """
        ε-greedy action selection.

        With probability ε — a random action (exploration)
        With probability (1-ε) — the best action (exploitation)
        """
        # TODO: implement ε-greedy
        # if self.rng.random() < epsilon:
        #     return a random action from 0 to num_actions-1
        # else:
        #     return argmax over the Q-table for state_idx
        ...  # ← your code

    def greedy_action(self, state_idx: int) -> int:
        """Greedy action selection (argmax Q)."""
        return int(np.argmax(self.q_table[state_idx]))

    def bellman_update(self, s_idx: int, action: int, reward: float,
                       next_idx: int, terminated: bool) -> None:
        """
        Q-value update using the Bellman equation.

        Q(s, a) ← Q(s, a) + α · [target - Q(s, a)]

        where target = r                           if terminated
                     = r + γ · max_a' Q(s', a')    otherwise
        """
        alpha = self.config.learning_rate
        gamma = self.config.discount

        # TODO: compute target
        # if terminated:
        #     target = reward  # no future — only the current reward
        # else:
        #     target = reward + gamma * max(Q[next_idx, :])  # Bellman equation
        target = ...  # ← your code

        # TODO: update the Q-table
        # td_error = target - Q[s_idx, action]
        # Q[s_idx, action] += alpha * td_error
        ...  # ← your code

    def train(self) -> Dict[str, List[float]]:
        """Main training loop."""
        metrics = {
            'episode_reward': [],
            'episode_length': [],
            'epsilon': []
        }

        for ep in tqdm(range(self.config.num_episodes), desc=f"Train {self.config.env_name}"):
            # Reset the environment
            obs, _ = self.env.reset()
            state_idx = self.state_index(obs)
            epsilon = self.epsilon_by_episode(ep)

            total_reward = 0.0
            steps = 0

            # Loop over episode steps
            for t in range(self.max_steps):
                # TODO: 1. Choose an action via ε-greedy
                action = ...  # ← your code

                # TODO: 2. Take a step in the environment
                next_obs, reward, terminated, truncated, _ = self.env.step(action)
                next_idx = self.state_index(next_obs)

                # TODO: 3. Update the Q-table (Bellman update)
                # Important: pass terminated, NOT (terminated or truncated)
                # truncated means a time-limit cutoff, not a terminal state
                ...  # ← your code

                # Update the counters
                total_reward += reward
                state_idx = next_idx
                steps += 1

                if terminated or truncated:
                    break

            # Save the metrics
            metrics['episode_reward'].append(total_reward)
            metrics['episode_length'].append(steps)
            metrics['epsilon'].append(epsilon)

        return metrics

    def evaluate(self, episodes: int = 5) -> Tuple[float, float]:
        """Evaluate the agent without training (greedy policy)."""
        env = gym.make(self.config.env_name)
        rewards, lengths = [], []

        try:
            for ep in range(episodes):
                obs, _ = env.reset(seed=self.config.seed + 10_000 + ep)
                state_idx = self.state_index(obs)
                total_reward = 0.0

                for t in range(self.max_steps):
                    # Greedy choice (no exploration)
                    action = self.greedy_action(state_idx)
                    obs, reward, terminated, truncated, _ = env.step(action)
                    total_reward += reward
                    state_idx = self.state_index(obs)

                    if terminated or truncated:
                        break

                rewards.append(total_reward)
                lengths.append(t + 1)
        finally:
            env.close()

        return float(np.mean(rewards)), float(np.mean(lengths))

### Checking the epsilon schedule

Before starting training it's useful to visualize how epsilon will change:

In [ ]:
# Visualizing the epsilon schedule
# Uncomment after implementing epsilon_by_episode

# # Create a temporary config
# test_config = QLearningConfig(
#     epsilon_start=1.0,
#     epsilon_end=0.05,
#     epsilon_decay_episodes=2000,
#     num_episodes=4000
# )
#
# # Create an agent for checking
# temp_env = gym.make('MountainCar-v0')
# temp_agent = QLearningAgent(temp_env, test_config)
#
# # Build the epsilon plot
# episodes = np.arange(test_config.num_episodes)
# epsilons = [temp_agent.epsilon_by_episode(ep) for ep in episodes]
#
# plt.figure(figsize=(10, 4))
# plt.plot(episodes, epsilons, 'b-', linewidth=2)
# plt.axhline(y=test_config.epsilon_end, color='r', linestyle='--', label=f'ε_end = {test_config.epsilon_end}')
# plt.axvline(x=test_config.epsilon_decay_episodes, color='g', linestyle='--', label=f'decay = {test_config.epsilon_decay_episodes}')
# plt.xlabel('Episode')
# plt.ylabel('Epsilon (ε)')
# plt.title('Linear epsilon schedule: from exploration to exploitation')
# plt.legend()
# plt.grid(True)
# plt.show()
#
# temp_env.close()
#
# # Value checks
# assert abs(temp_agent.epsilon_by_episode(0) - 1.0) < 0.01, "ε(0) should be ~1.0"
# assert abs(temp_agent.epsilon_by_episode(2000) - 0.05) < 0.01, "ε(2000) should be ~0.05"
# assert temp_agent.epsilon_by_episode(3000) >= 0.05, "ε should not drop below ε_end"
# print("✅ Epsilon schedule works correctly!")

## 5. Evaluating the greedy policy (standalone function)
Sometimes it's convenient to separate the evaluation function from the class. Implement a helper function that runs N
episodes without `epsilon` and returns the average rewards/lengths. Use it in the experiments to validate the models.

In [ ]:
def evaluate_policy(env_name: str, agent: QLearningAgent, episodes: int = 5) -> Tuple[float, float]:
    """
    Evaluates the agent's greedy policy in a separate environment.

    Returns:
        (average reward, average episode length)
    """
    env = gym.make(env_name)
    rewards, lengths = [], []

    try:
        for ep in range(episodes):
            obs, _ = env.reset(seed=agent.config.seed + 20_000 + ep)
            state_idx = agent.state_index(obs)
            total_reward = 0.0

            for t in range(agent.max_steps):
                # TODO: choose the greedy action and take the step
                action = ...  # ← use agent.greedy_action(state_idx)
                obs, reward, terminated, truncated, _ = env.step(action)

                total_reward += reward
                state_idx = agent.state_index(obs)

                if terminated or truncated:
                    break

            rewards.append(total_reward)
            lengths.append(t + 1)
    finally:
        env.close()

    return float(np.mean(rewards)), float(np.mean(lengths))

In [ ]:
def moving_average(values: List[float], window: int = 100) -> np.ndarray:
    """Computes the moving average for smoothing plots."""
    arr = np.asarray(values, dtype=np.float32)
    if arr.size == 0:
        return arr
    if arr.size < window:
        return arr
    kernel = np.ones(window, dtype=np.float32) / window
    return np.convolve(arr, kernel, mode='valid')

## 6. Experiment A — `MountainCar`: the effect of discretization
1. Use the same training configuration but different bin grids (e.g., `(18, 18)` vs `(30, 30)`).
2. Fix the seeds for a fair comparison.
3. Log the moving average of the reward, episode lengths, and the final greedy score.
4. Draw conclusions about how many bins are needed for stable learning in `MountainCar`.

In [ ]:
# Experiment A: The effect of discretization on MountainCar
# Comparing a coarse and a fine bin grid

mc_results: Dict[str, Dict[str, object]] = {}

# Configurations to compare
mc_configs = {
    '20x20 (coarse)': {
        'bins': (20, 20),
        'num_episodes': 2000,
        'epsilon_decay': 1000,
        'learning_rate': 0.2,
    },
    '30x30 (fine)': {
        'bins': (30, 30),
        'num_episodes': 3000,
        'epsilon_decay': 1500,
        'learning_rate': 0.25,
    },
}

for label, cfg in mc_configs.items():
    print(f"\n{'='*50}")
    print(f"Training: {label}")
    print(f"{'='*50}")

    # TODO: create the environment
    env = gym.make('MountainCar-v0')

    # TODO: create the configuration
    # config = QLearningConfig(
    #     env_name='MountainCar-v0',
    #     bins=cfg['bins'],
    #     num_episodes=cfg['num_episodes'],
    #     epsilon_decay_episodes=cfg['epsilon_decay'],
    #     learning_rate=cfg['learning_rate'],
    #     discount=0.99,
    #     seed=SEED,
    # )
    config = ...  # ← your code

    # TODO: create and train the agent
    # agent = QLearningAgent(env, config)
    # logs = agent.train()
    agent = ...  # ← your code
    logs = ...   # ← your code

    # TODO: evaluate the agent
    # eval_reward, eval_length = agent.evaluate(episodes=10)
    eval_reward, eval_length = ...  # ← your code

    # Save the results
    mc_results[label] = {
        'logs': logs,
        'eval_reward': eval_reward,
        'eval_length': eval_length,
        'bins': cfg['bins'],
    }

    env.close()
    n_states = cfg['bins'][0] * cfg['bins'][1]
    print(f"Result: eval={eval_reward:.1f}, length={eval_length:.1f}, states={n_states}")

In [ ]:
# Visualizing the results of Experiment A

if mc_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for label, result in mc_results.items():
        rewards = result['logs']['episode_reward']
        lengths = result['logs']['episode_length']

        # TODO: compute the moving average for smoothing
        # ma_rewards = moving_average(rewards, window=50)
        # ma_lengths = moving_average(lengths, window=50)
        ma_rewards = ...  # ← your code
        ma_lengths = ...  # ← your code

        # TODO: build the plots
        # axes[0].plot(ma_rewards, label=f"{label} (eval {result['eval_reward']:.0f})")
        # axes[1].plot(ma_lengths, label=label)
        ...  # ← your code

    # Configure the plots
    axes[0].set_title('MountainCar: reward (MA 50)')
    axes[0].set_xlabel('Episodes')
    axes[0].set_ylabel('Reward')
    axes[0].grid(True)
    axes[0].legend()

    axes[1].set_title('MountainCar: episode length (MA 50)')
    axes[1].set_xlabel('Episodes')
    axes[1].set_ylabel('Steps')
    axes[1].grid(True)
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print('Run the Experiment A cell first.')

### Conclusions for Experiment A

**Answer the following questions:**

1. **Convergence speed:** Which bin grid learned faster? Why?
   - TODO: your answer

2. **Final performance:** Which configuration achieved the best final evaluation reward?
   - TODO: your answer

3. **Stability:** Which grid showed less fluctuation in reward during the later stages of training?
   - TODO: your answer

4. **Recommendations:** How many bins would you recommend for MountainCar and why?
   - TODO: your answer

## 7. Experiment B — `Acrobot`: epsilon schedules and stability
1. Fix a single discretization (e.g., the default one) and compare at least two `epsilon` schedules
   (fast and slow decay).
2. Analyze how often the agent reaches the target height and what rewards it gets.
3. Compare the variance of the rewards and draw conclusions about which exploration strategy is better for `Acrobot`.

In [ ]:
# Experiment B: The effect of the epsilon schedule on Acrobot
# Comparing fast and slow epsilon decay

acrobot_results: Dict[str, Dict[str, object]] = {}

# Schedules to compare
acrobot_schedules = {
    'fast_decay (ε→0.05 over 600 ep.)': {
        'decay': 600,
        'num_episodes': 5000,
    },
    'slow_decay (ε→0.05 over 2500 ep.)': {
        'decay': 2500,
        'num_episodes': 5000,
    },
}

for label, cfg in acrobot_schedules.items():
    print(f"\n{'='*50}")
    print(f"Training: {label}")
    print(f"{'='*50}")

    env = gym.make('Acrobot-v1')

    # TODO: create the configuration for Acrobot
    # Recommended parameters:
    # - learning_rate = 0.2
    # - discount = 0.995 (higher than for MountainCar)
    # - bins = None (uses default_bins from EnvSpec)
    config = QLearningConfig(
        env_name='Acrobot-v1',
        num_episodes=cfg['num_episodes'],
        epsilon_decay_episodes=cfg['decay'],
        learning_rate=0.2,
        discount=0.995,
        seed=SEED + 7,
    )

    # TODO: train the agent
    agent = ...  # ← your code
    logs = ...   # ← your code

    # Evaluation
    eval_reward, eval_length = agent.evaluate(episodes=10)

    # Compute success_rate: fraction of episodes that finished before max_steps
    lengths = np.asarray(logs['episode_length'])
    success_rate = float(np.mean(lengths < agent.max_steps))

    acrobot_results[label] = {
        'logs': logs,
        'eval_reward': eval_reward,
        'eval_length': eval_length,
        'success_rate': success_rate,
    }

    env.close()
    print(f"Result: eval={eval_reward:.1f}, success_rate={success_rate:.2%}")

In [ ]:
# Visualizing the results of Experiment B

if acrobot_results:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for label, result in acrobot_results.items():
        rewards = result['logs']['episode_reward']
        lengths = result['logs']['episode_length']
        epsilons = result['logs']['epsilon']

        # Use a larger smoothing window due to high variance
        ma_rewards = moving_average(rewards, window=200)
        ma_lengths = moving_average(lengths, window=200)

        # TODO: build the plots for rewards, lengths, and epsilon
        axes[0].plot(ma_rewards, label=f"{label} (eval {result['eval_reward']:.0f})")
        axes[1].plot(ma_lengths, label=label)
        axes[2].plot(epsilons, label=label)

    # Configure the plots
    axes[0].set_title('Acrobot: reward (MA 200)')
    axes[0].set_xlabel('Episodes')
    axes[0].set_ylabel('Reward')
    axes[0].grid(True)
    axes[0].legend()

    axes[1].set_title('Acrobot: episode length (MA 200)')
    axes[1].set_xlabel('Episodes')
    axes[1].set_ylabel('Steps')
    axes[1].grid(True)
    axes[1].legend()

    axes[2].set_title('Epsilon schedule')
    axes[2].set_xlabel('Episodes')
    axes[2].set_ylabel('ε')
    axes[2].grid(True)
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    # Results table
    print("\nResults summary:")
    print("-" * 60)
    for label, result in acrobot_results.items():
        print(f"{label}:")
        print(f"  Eval reward: {result['eval_reward']:.1f}")
        print(f"  Success rate: {result['success_rate']:.2%}")
else:
    print('Run the Experiment B cell first.')

### Conclusions for Experiment B

**Answer the following questions:**

1. **Effect of the epsilon schedule:** How did the epsilon decay speed affect training?
   - TODO: your answer

2. **Success rate:** Which schedule led to a higher fraction of successful episodes? Why?
   - TODO: your answer

3. **Exploration vs exploitation:** Explain the balance between exploration and exploitation in each schedule.
   - TODO: your answer

4. **Recommendations:** Which epsilon schedule would you choose for Acrobot with a limited episode budget (e.g., 3000)?
   - TODO: your answer

## 8. Additional investigations (optional)

If you want to dig deeper into the topic, try the following experiments:

### Idea 1: Adaptive discretization
- Use a non-uniform bin grid (more bins in critical regions of the state space)
- For MountainCar: more bins around velocity = 0 and position = -0.5

### Idea 2: Adding a third environment
- Add `CartPole-v1` or `Pendulum-v1` (with action discretization)
- Create a new EnvSpec and check the generality of your code

### Idea 3: Analyzing the Q-table
- Visualize Q-values for different states
- Find the "key" states where the agent makes critical decisions
- For MountainCar, build a heatmap of Q(position, velocity, action)

### Idea 4: Combined epsilon schedule
- Try exponential decay instead of linear
- Or a two-phase schedule: fast down to 0.3, then slow down to 0.05

### Idea 5: Effect of the learning rate
- Compare different values of α (0.05, 0.1, 0.2, 0.5)
- Plot how convergence speed depends on α

**Your experiments:**
TODO: describe what you tried and what conclusions you reached

## 9. Self-check questions

1. **TODO:** How do the discretization requirements differ for MountainCar (2D) and Acrobot (6D)? Which dimensions require finer discretization and why?

2. **TODO:** Which epsilon schedule performed best in each environment and why? Is there a universal strategy, or does it need to be adapted per task?

3. **TODO:** Which metrics did you use to monitor convergence and stability? Why isn't it enough to look only at the average reward?

4. **TODO:** What problems did you run into when applying tabular Q-learning? Which environments would not be suitable for this approach?